## Reading Bronze.Airlines Delta Table

In [0]:
Airlines_bronze_path = "s3://travel-analytics-bronze/delta/bronze/airlines/"
Airlines_bronze_df = spark.read.format("delta").load(Airlines_bronze_path)

## Silver Transformations

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
col, trim, upper, to_date, to_timestamp,
    when, date_format, concat, lit, coalesce, expr ,initcap
)

# =============================================================
# STEP 0:CONFIGURATION & SETUP
# =============================================================
table_name = "airlines"
airlines_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"

# =============================================================
# STEP 1: DATA TYPE CASTING & Parsing
# =============================================================
print("\nSTEP 1: Casting Data Types (Airlines)...")

airlines_step_1_df = (
    Airlines_bronze_df

    # ========== Numeric columns ==========
    .withColumn("airline_id", col("airline_id").cast("int"))
    .withColumn("fleet_size", col("fleet_size").cast("int"))

    # ========== String columns ==========
    .withColumn("airline_name", initcap(trim(col("airline_name"))))
    .withColumn("alias", initcap(trim(col("alias"))))
    .withColumn("country", initcap(trim(col("country"))))
    .withColumn("airline_iata", trim(upper(col("airline_iata"))))
    .withColumn("airline_icao", trim(upper(col("airline_icao"))))

    # ========== CDC updated timestamp ==========
    .withColumn("updated_at", to_timestamp(col("_ab_cdc_updated_at")))
)
# ===================================================================================
# STEP 2: Cleaning , Standardizing String Columns and  Business Logic applications
# ===================================================================================
print("\nSTEP 2: Cleaning & Standardization (Airlines)...")

airlines_step_2_df = (
    airlines_step_1_df

    # Handle nulls
    .withColumn("airline_name", coalesce(col("airline_name"), lit("UNKNOWN")))
    .withColumn("alias", coalesce(col("alias"), lit("UNKNOWN")))
    .withColumn("country", coalesce(col("country"), lit("UNKNOWN")))
    .withColumn("airline_iata", coalesce(col("airline_iata"), lit("UNKNOWN")))
    .withColumn("airline_icao", coalesce(col("airline_icao"), lit("UNKNOWN")))
    
)
# ===================================================================================
# STEP 3: DEDUPLICATION & DROP AIRBYTE METADATA
# ===================================================================================
print("\nSTEP 3: Deduplication & Dropping Airbyte Columns (Airlines)...")

airbyte_columns_to_drop = [
    "_airbyte_ab_id",
    "_airbyte_emitted_at",
    "_airbyte_additional_properties",
    "_ab_cdc_lsn",
    "_ab_cdc_deleted_at",
]

airlines_silver_df = (
    airlines_step_2_df

    # Deduplicate on natural key
    .dropDuplicates(["airline_id"])

    # Drop metadata
    .drop(*airbyte_columns_to_drop)
)



STEP 1: Casting Data Types (Airlines)...

STEP 2: Cleaning & Standardization (Airlines)...

STEP 3: Deduplication & Dropping Airbyte Columns (Airlines)...


In [0]:
print("\nSTEP 4: Renaming Columns (Airlines)...")

rename_map = {
    "airline_id": "Airline_Id",
    "airline_name": "Airline_Name",
    "alias": "Alias",
    "country": "Country",
    "fleet_size": "Fleet_Size",
    "airline_iata": "Airline_IATA",
    "airline_icao": "Airline_ICAO",
    "updated_at": "Updated_At",
}

airlines_silver_df = airlines_silver_df.select(
    [col(c).alias(rename_map.get(c, c)) for c in airlines_silver_df.columns]
)



STEP 4: Renaming Columns (Airlines)...


In [0]:
airlines_silver_df.display()

Alias,Country,Airline_Id,Fleet_Size,Airline_IATA,Airline_ICAO,Airline_Name,_ab_cdc_updated_at,Updated_At
UNKNOWN,South Africa,3,8,1T,RNX,1time Airline,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,United States,10,4,Q5,MLA,40-mile Air,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,Australia,13,45,AN,AAA,Ansett Australia,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,Singapore,14,0,1B,UNKNOWN,Abacus International,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,France,21,12,ZI,AAF,Aigle Azur,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,United States,22,18,AQ,AAH,Aloha Airlines,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,United States,24,875,AA,AAL,American Airlines,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,Republic Of Korea,28,82,OZ,AAR,Asiana Airlines,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,Pakistan,29,6,4K,AAS,Askari Aviation,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z
UNKNOWN,Libya,32,13,8U,AAW,Afriqiyah Airways,2025-12-12T01:04:52.921893877Z,2025-12-12T01:04:52.921Z


## Writing Silver airlines to Delta Lake 

In [0]:
print("\nSTEP 5: Persist Airlines Silver Table...")

airlines_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(airlines_silver_path)


STEP 5: Persist Airlines Silver Table...
